# Create files mapping FACS well and plate to basic allele metadata

This script takes cherry picking and allele info files as input and produces a table relating FACS well and plate IDs to allele IDs and symbols. 

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os

In [2]:
dualipa_inputs = "../1_inputs"
dualipa_outputs = "../3_outputs"
allele_collection_inputs = "../../../1_allele_collection/1_inputs/seq_confirmation_results"
allele_collection_outputs = "../../../1_allele_collection/3_outputs"

## 1. Loading the pdest layout data frame

Vidal lab ran experiment once and analyzed the data. Some alleles didn't pass the qa/qc thresholds or got no data. Batch5 is is a re-run for all alleles that failed in the first set of batches. Here, we process everything together from scratch. An allele should always be compared to the wt on the same batch and plate. The summary stats can then be compared across batches. 

In [3]:
# Read in cherry picking files for both the initial batches and the second round
cp_col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']

pdest_layout_initial_batches_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP.csv', index_col=0)[cp_col]
pdest_layout_b5_df = pd.read_csv(f'{dualipa_inputs}/cherry_picking_files/summary_CP_VUSDUALXP_batch5.csv', index_col=0)[cp_col]

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_initial_batches_df])
pdest_layout_df = pl.DataFrame(pdest_layout_df)
pdest_layout_df = pdest_layout_df.with_columns(
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.col("mut_id").cast(pl.Int64).alias("mut_id")
)
pdest_layout_df
### This doesn't have well A01

orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well
i64,i64,str,i64,str,str
56360,0,"""wt""",23,"""VUSDUALXP_23""","""A12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""B12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""C12"""
56360,0,"""wt""",23,"""VUSDUALXP_23""","""D12"""
9905,0,"""wt""",23,"""VUSDUALXP_23""","""F02"""
…,…,…,…,…,…
null,null,"""pdest_empty""",20,"""VUSDUALXP_20""","""A08"""
null,null,"""pdest_empty""",21,"""VUSDUALXP_21""","""A07"""
null,null,"""pdest_empty""",21,"""VUSDUALXP_21""","""A08"""


In [4]:
pdest_layout_df.filter((pl.col("dest_pla_id")==18)&(pl.col("dest_well")=="E11"))

orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well
i64,i64,str,i64,str,str
3438,201180,"""allele""",18,"""VUSDUALXP_18""","""E11"""


## 2. Loading the sequence confirmed results

In [5]:
seq_confirm_code_df = pd.read_table(f"{allele_collection_inputs}/sequence_confirmation_class_code.tsv")
with pd.option_context('display.max_columns', None):
    display(seq_confirm_code_df)

,code,description
0,1,perfectly validated
1,2,"partially validated, >= 50% coverage"
2,3,wild type
3,4,"partial wild type, >= 50% coverage"
4,5,"target and off-target mutation, >= 50% coverage"
5,6,"off-target mutation, >= 50% coverage"
6,7,"truncated, < 50% coverage"
7,99,no reads


In [6]:
seq_confirm_res_df = pl.read_parquet(
    f"{allele_collection_outputs}/VarChAMP_Allele_Collection_1Perc_ClinVar_dbNSFP_pLDDT_RSA_G2P_OMIM_MOI.parquet"
).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id")
)

with pl.Config(set_tbl_cols=100):
    display(seq_confirm_res_df)

symbol,aa_change,gene_variant,spdi,nt_change,chr_num,nuc_loc,ref_allele,alt_allele,Chrom,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,seq,seq_aa,ensembl_protein_id,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,entry_seq_final,db_seq_final,n2h_seq_final,dualip_seq_final,mislocalization_seq_final,…,g2p_Intra-chain Non-bonded interaction (AlphaFold2),g2p_Intra-chain Non-bonded interaction (PDB),g2p_Intra-chain Salt bridge (AlphaFold2),g2p_Intra-chain Salt bridge (PDB),g2p_Intramembrane (UniProt),g2p_Lipidation (UniProt),g2p_Methylation,g2p_Modified residue (UniProt),g2p_Molar mass (g/mol),g2p_Motif (UniProt),g2p_Mutagenesis (UniProt),g2p_Non-adjacent residues (UniProt),g2p_Non-standard residue (UniProt),g2p_Non-terminal residue (UniProt),g2p_O-GalNAc,g2p_O-GlcNAc,g2p_Peptide (UniProt),g2p_Phi angle (degrees)*,g2p_Phosphorylation,g2p_Pocket number (fpocket)*,g2p_Propeptide (UniProt),g2p_Psi angle (degrees)*,g2p_Region (UniProt),g2p_Regulatory sites,g2p_Repeat (UniProt),g2p_SNP-associated PTMs,g2p_SUMOylation,g2p_Secondary structure (DSSP 3-state)*,g2p_Secondary structure (DSSP 9-state)*,g2p_Secondary structure (PDBe/SIFTS),g2p_Sequence conflict (UniProt),g2p_Sequence uncertainty (UniProt),g2p_Signal (UniProt),g2p_Site (UniProt),g2p_Substrate genes,g2p_Topological domain (UniProt),g2p_Transit peptide (UniProt),g2p_Transmembrane (UniProt),g2p_Ubiquitination,g2p_Zinc finger (UniProt),plddt_f32,rsa_f32,pLDDT_Category,RSA_Category,OMIM_IDs,inheritance_pattern,inheritance_source,disease_modules,disease_module_count,OMIM_disease_names
str,str,str,str,str,i32,i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f32,f32,str,str,str,str,str,str,i64,str
"""GBA1""","""Val54Leu""","""GBA1_Val54Leu""","""NC_000001.11:155240033:C:G""","""160G>C""",1,155240033,"""C""","""G""","""1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""MEFSSPSREECPKPLSRVSIMAGSLTGLLL…","""ENSP00000357357.3""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1.0,1.0,1.0,2.0,0.0,null,null,0.0,null,null,0.0,null,null,"""Y""","""Y""","""N""","""N""","""N""",…,""" V54-G394 (PAE: 1.0), V54-K46…","""V54-K46 (PDBs: 1OGS, 2J25, 2NS…","""""","""""","""""","""""","""""","""""","""117""","""""","""""","""""","""""","""""","""""","""""","""""","""-86.4""","""""","""""","""""","""160.1""","""""","""""","""""","""""","""""","""B (undefined)""","""E (parallel sheets)""","""Beta strand""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",null,null,"""NaN""","""NaN""",null,null,null,null,0,null
"""GBA1""","""Leu224Phe""","""GBA1_Leu224Phe""","""NC_000001.11:155238225:G:A""","""670C>T""",1,155238225,"""G""","""A""","""1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…","""MEFSSPSREECPKPLSRVSIMAGSLTGLLL…","""ENSP00000357357.3""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""

## 3. Map the pdest mutation to meta mutation information

In [7]:
# seq_confirm_res_df.filter(pl.col("mut_id")==0)

## map orf_id to symbol and ensembl_gene_id
orfid_2_symbol_mapping = dict(zip(
    seq_confirm_res_df.filter(pl.col("symbol").is_not_null())["orf_id"].to_list(),
    seq_confirm_res_df.filter(pl.col("symbol").is_not_null())["symbol"].to_list()
))

orfid_2_geneid_mapping = dict(zip(
    seq_confirm_res_df.filter(pl.col("ensembl_gene_id").is_not_null())["orf_id"].to_list(),
    seq_confirm_res_df.filter(pl.col("ensembl_gene_id").is_not_null())["ensembl_gene_id"].to_list()
))

In [8]:
## join the pdest variant with mut meta info by "INNER" JOIN;
## some alleles were filtered out during the allele collection stage due to their poor seq quality
## SHOULD REMOVE them from DUAL-IPA assay
pdest_layout_seq_conf_mut_df = pdest_layout_df.filter(
    pl.col("mut_id")!=0
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="inner"
)

## need to add wt data here as well
pdest_layout_seq_conf_wt_df = pdest_layout_df.filter(
    pl.col("mut_id")==0
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="left"
).with_columns(
    pl.col("aa_change").fill_null("WT"),
    pl.col("nt_change").fill_null("WT"),
    pl.col("orf_id").map_elements(lambda x: orfid_2_symbol_mapping[x] if x in orfid_2_symbol_mapping else pl.Null, return_dtype=pl.String).alias("symbol"),
    pl.col("orf_id").map_elements(lambda x: orfid_2_geneid_mapping[x] if x in orfid_2_geneid_mapping else pl.Null, return_dtype=pl.String).alias("ensembl_gene_id")
)

## need to add the pdest_empty wells as well
pdest_layout_seq_conf_empty_df = pdest_layout_df.filter(
    pl.col("mut_id").is_null()
).join(
    seq_confirm_res_df, 
    on=["orf_id", "mut_id"],
    how="left"
)

## concat var and wt and empty
pdest_layout_seq_conf_df = pl.concat([
    pdest_layout_seq_conf_mut_df, pdest_layout_seq_conf_wt_df, pdest_layout_seq_conf_empty_df
])
print(pdest_layout_seq_conf_df["dualip_sequence_confirmation_class"].unique())

shape: (9,)
Series: 'dualip_sequence_confirmation_class' [f64]
[
	null
	1.0
	2.0
	3.0
	4.0
	5.0
	6.0
	7.0
	99.0
]


In [9]:
pdest_layout_seq_conf_df.unique(subset="dualip_seq_final")["dualip_seq_final"]

dualip_seq_final
str
null
"""N"""
"""Y"""


In [10]:
pdest_layout_seq_conf_df

orf_id,mut_id,node_type,dest_pla_id,dest_pla,dest_well,symbol,aa_change,gene_variant,spdi,nt_change,chr_num,nuc_loc,ref_allele,alt_allele,Chrom,ensembl_gene_id,ccsb_mutation_id,ccsb_allele_id,seq,seq_aa,ensembl_protein_id,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,…,g2p_Non-terminal residue (UniProt),g2p_O-GalNAc,g2p_O-GlcNAc,g2p_Peptide (UniProt),g2p_Phi angle (degrees)*,g2p_Phosphorylation,g2p_Pocket number (fpocket)*,g2p_Propeptide (UniProt),g2p_Psi angle (degrees)*,g2p_Region (UniProt),g2p_Regulatory sites,g2p_Repeat (UniProt),g2p_SNP-associated PTMs,g2p_SUMOylation,g2p_Secondary structure (DSSP 3-state)*,g2p_Secondary structure (DSSP 9-state)*,g2p_Secondary structure (PDBe/SIFTS),g2p_Sequence conflict (UniProt),g2p_Sequence uncertainty (UniProt),g2p_Signal (UniProt),g2p_Site (UniProt),g2p_Substrate genes,g2p_Topological domain (UniProt),g2p_Transit peptide (UniProt),g2p_Transmembrane (UniProt),g2p_Ubiquitination,g2p_Zinc finger (UniProt),plddt_f32,rsa_f32,pLDDT_Category,RSA_Category,OMIM_IDs,inheritance_pattern,inheritance_source,disease_modules,disease_module_count,OMIM_disease_names
i64,i64,str,i64,str,str,str,str,str,str,str,i32,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f32,f32,str,str,str,str,str,str,i64,str
281,727,"""allele""",22,"""VUSDUALXP_22""","""E09""","""MVK""","""His20Gln""","""MVK_His20Gln""","""NC_000012.12:109574882:T:A""","""60T>A""",12,109574882,"""T""","""A""","""12""","""ENSG00000110921""","""CCSBVarC000026""","""ALE00000026""","""ATGTTGTCAGAAGTCCTACTGGTGTCTGCT…","""MLSEVLLVSAPGKVILHGEQAVVHGKVALA…","""ENSP00000228510.3""","""RC4""","""RC4_Mut_GDEh1020""","""B03""","""GDEhDisVCh_40020""","""F04""","""2""","""RC4_Mut_GDDh1020""","""B03""","""VUSN2Hmut_GDN2h_10""","""D08""","""VUSMutpDEST2_01""","""B04""","""VUSmut_GDEh13""","""E03""",1.0,…,"""""","""""","""""","""""","""-92.8""","""""","""Pocket 2: Mean pLDDT: 90.7, Vo…","""""","""-21.4""","""""","""""","""""","""""","""""","""C (loop/coil)""","""T (turn)""","""Turn""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",94.910004,4.347826,""">90""","""<10""","""260920;175900;610377;260920""","""autosomal recessive""","""OMIM""","""Rare genetic systemic or rheum…",6,"""HYPER-IgD SYNDROME; POROKERATO…"
281,13,"""allele""",22,"""VUSDUALXP_22""","""E07""","""MVK""","""Pro167Leu""","""MVK_Pro167Leu""","""NC_000012.12:109581523:C:T""","""500C>T""",12,109581523,"""C""","""T""","""12""","""ENSG00000110921""","""CCSBVarC000027""","""ALE00000027""","""ATGTTGTCAGAAGTCCTACTGGTGTCTGCT…","""MLSEVLLVSAPGKVILHGEHAVVHGKVALA…","""ENSP00000228510.3""","""Edgotyping3""","""VUSMutGDEh009""","""C12""","""NULL""","""NULL""","""NULL""","""VUSMutGDDh009""","""C12""","""VUSN2Hmut_GDN2h_03""","""H04""","""VUSMutpDEST2_01""","""C02""","""VUSmut_GDEh03""","""H04""",0.0,…,"""""","""""","""""","""""","""-82.3""","""""","""""","""""","""-10.4""","""""","""""","""""","""""","""""","""C (loop/coil)""","""C (loop/coil)""","""Turn""","""""","""""","""""","""""","""""","""""","""""","""""","""""","""""",93.32,13.235294,""">90""","""10-90""","""260920;260920;610377;175900""","""autosomal recessive""","""OMIM""","""Rare genetic systemic or rheum…",6,"""HYPER-IgD SYNDROME; HYPER-IgD …"
281,13,"""allele""",22,"""VUSDUALXP_22""","""E07""","""MVK""","""Pro167Leu""","""MVK_Pro167Leu""","""NC_000012.12:109581523:C:T""","""500C>T""",12,109581523,"""C""","""T""","""12""","""ENSG00000110921""","""CCSBVarC000027""","""ALE00000027""","""ATGTTGTCAGAAGTCCTACTGGTGTCTGCT…","""MLSEVLLVSAPGKVILHGEHAVVHGKVALA…","""ENSP00000228510.3""","""RC4""","""RC4_Mut_GDEh1020""","""C03""","""GDEhDisVCh_40020""","""G04""","""2""","""RC4_Mut_GDDh1020""","""C03""","""NULL""

In [18]:
pdest_layout_seq_conf_df[pdest_layout_seq_conf_df.columns[:53]].write_csv(f"{dualipa_outputs}/pdest_layout_DACC.tsv")

In [11]:
pdest_layout_seq_conf_df.write_parquet(f'{dualipa_outputs}/dualipa_experimental_layout.parquet')